# Власна RAG-система: PHP-помічник
Авторські навчальні нотатки, створені для експерименту (CC0). Призначення — пошук довідок для студентів. RAG дозволяє оновлювати джерела без навчання моделі та цитувати їх.

In [1]:
%pip -q install sentence-transformers faiss-cpu rank-bm25 gradio

In [2]:
import torch, numpy as np, pandas as pd, re, faiss, time
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display
start=time.perf_counter()
items=[('PDO', 'PDO::prepare', 'Підготовлений SQL-запит створює PDO::prepare; параметри передаються через execute.', 'Як підготувати SQL-запит?'), ('PDO', 'PDO::beginTransaction', 'Транзакцію починає PDO::beginTransaction, підтверджує commit, скасовує rollBack.', 'Як почати транзакцію?'), ('PDO', 'utf8mb4', 'У DSN MySQL charset=utf8mb4 задає кодування з підтримкою Unicode.', 'Яке кодування задавати в DSN?'), ('Безпека', 'password_hash', 'Паролі зберігають як хеш password_hash, перевіряють password_verify.', 'Як хешувати пароль?'), ('Безпека', 'htmlspecialchars', 'htmlspecialchars екранує спеціальні символи HTML при виведенні тексту.', 'Як екранувати HTML?'), ('Безпека', 'random_bytes', 'CSRF-токен можна створити random_bytes і перевірити hash_equals.', 'Як створити випадковий CSRF-токен?'), ('Сесії', 'session_start', 'session_start запускає або відновлює сесію до виведення заголовків.', 'Як почати сесію?'), ('Сесії', 'session_regenerate_id', 'Після входу session_regenerate_id оновлює ідентифікатор сесії.', 'Як оновити ID сесії після входу?'), ('Сесії', '$_SESSION', 'Дані сесії доступні через масив $_SESSION.', 'Де зберігаються змінні сесії у коді?'), ('HTTP', 'header', 'header надсилає HTTP-заголовок до виведення тіла; після перенаправлення варто викликати exit.', 'Яка функція надсилає заголовок?'), ('HTTP', '$_POST', 'Поля HTML-форми з методом POST доступні через $_POST.', 'Де отримати поля POST-форми?'), ('HTTP', '$_GET', 'Параметри рядка запиту доступні через $_GET.', 'Де отримати параметри URL?'), ('OOP', 'private', 'Модифікатор private обмежує доступ до члена класу самим класом.', 'Який модифікатор приховує властивість від інших класів?'), ('OOP', 'extends', 'Клас успадковує інший клас за допомогою extends.', 'Як успадкувати клас?'), ('OOP', 'implements', 'Клас реалізує інтерфейс за допомогою implements.', 'Як реалізувати інтерфейс?'), ('Composer', 'composer install', 'composer install встановлює залежності відповідно до composer.lock, якщо він існує.', 'Як встановити зафіксовані залежності?'), ('Composer', 'vendor/autoload.php', 'Файл vendor/autoload.php підключає автозавантажувач Composer.', 'Який файл підключає автозавантаження?'), ('Файли', '__DIR__', 'Константа __DIR__ повертає каталог поточного PHP-файлу.', 'Як отримати каталог поточного файлу?'), ('Файли', 'require_once', 'require_once підключає файл один раз і зупиняє виконання при невдачі.', 'Як обов’язково підключити файл один раз?'), ('JSON', 'json_decode', 'json_decode перетворює JSON на PHP-значення; true другим аргументом дає асоціативні масиви для об’єктів.', 'Як декодувати JSON?')]
bench=[(q,i,a) for i,(d,a,t,q) in enumerate(items)]+[(q,None,None) for q in ["Яка ціна курсу PHP?","Який розклад університету?","Яка версія Laravel вийде наступного року?","Хто автор PHP?","Який тариф хостингу обрати?"]]
chunks=[f"{d}. {t}" for d,a,t,q in items]
print("Документів:",len(set(x[0] for x in items)),"фрагментів:",len(chunks),"символів:",sum(map(len,chunks)))
display(pd.DataFrame(bench,columns=["Питання","Фрагмент (0-based)","Еталон"]))

Документів: 8 фрагментів: 20 символів: 1457


,Питання,Фрагмент (0-based),Еталон
0,Як підготувати SQL-запит?,0.0,PDO::prepare
1,Як почати транзакцію?,1.0,PDO::beginTransaction
2,Яке кодування задавати в DSN?,2.0,utf8mb4
3,Як хешувати пароль?,3.0,password_hash
4,Як екранувати HTML?,4.0,htmlspecialchars
5,Як створити випадковий CSRF-токен?,5.0,random_bytes
6,Як почати сесію?,6.0,session_start
7,Як оновити ID сесії після входу?,7.0,session_regenerate_id
8,Де зберігаються змінні сесії у коді?,8.0,$_SESSION
9,Яка функція надсилає заголовок?,9.0,header


## Індексація та ретрив
Розбиття за завершеними тематичними абзацами, перекриття 0; FAISS inner product для нормалізованих векторів. Порівнюємо семантичний пошук і RRF з BM25.

In [3]:
device="cuda" if torch.cuda.is_available() else "cpu"
emb=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",device=device)
v=emb.encode(chunks,normalize_embeddings=True).astype("float32")
idx=faiss.IndexFlatIP(v.shape[1]); idx.add(v)
tok=lambda s:re.findall(r"\w+",s.lower())
bm=BM25Okapi([tok(c) for c in chunks])
qv=emb.encode([q for q,_,_ in bench],normalize_embeddings=True).astype("float32")
def rank(q,vec,hybrid=False):
    ids=idx.search(vec.reshape(1,-1),len(chunks))[1][0].tolist()
    if not hybrid:return ids
    lex=np.argsort(-bm.get_scores(tok(q))).tolist()
    return sorted(ids,key=lambda i:1/(61+ids.index(i))+1/(61+lex.index(i)),reverse=True)
for h in [False,True]:
    ranks=[rank(q,vec,h).index(g)+1 for (q,g,a),vec in zip(bench,qv) if g is not None]
    print("hybrid",h,"hit@1",np.mean(np.array(ranks)<=1),"hit@5",np.mean(np.array(ranks)<=5),"MRR",np.mean(1/np.array(ranks)))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

hybrid False hit@1 0.9 hit@5 0.95 MRR 0.9321428571428572
hybrid True hit@1 0.95 hit@5 1.0 MRR 0.9666666666666666


## Генерація
Компактна Qwen2.5-1.5B-Instruct, локальний inference у Colab. Це прямий виклик Transformers, не OpenAI-сумісний API; за потреби суворої відповідності методичці використати ендпойнт лабораторної 4.

In [4]:
name="Qwen/Qwen2.5-1.5B-Instruct"
t=AutoTokenizer.from_pretrained(name); t.padding_side="left"
m=AutoModelForCausalLM.from_pretrained(name,torch_dtype=torch.float16 if device=="cuda" else torch.float32).to(device).eval()
def prompts(q,ids):
    context="\n".join(f"[{i+1}] {chunks[i]}" for i in ids)
    return t.apply_chat_template([{"role":"system","content":"Відповідай українською одним реченням лише за контекстом. Вкажи номер джерела [N]. Якщо відповіді немає, напиши НЕМАЄ ВІДПОВІДІ."},{"role":"user","content":context+"\nПитання: "+q}],tokenize=False,add_generation_prompt=True)
def generate(ps):
    x=t(ps,padding=True,return_tensors="pt").to(device)
    with torch.inference_mode(): y=m.generate(**x,max_new_tokens=64,do_sample=False,pad_token_id=t.eos_token_id)
    return t.batch_decode(y[:,x.input_ids.shape[1]:],skip_special_tokens=True)
rows=[]
for offset in range(0,len(bench),5):
    batch=bench[offset:offset+5]; ids=[rank(q,vec,True)[:3] for (q,g,a),vec in zip(batch,qv[offset:offset+5])]
    answers=generate([prompts(q,r) for (q,g,a),r in zip(batch,ids)])
    for (q,g,a),r,s in zip(batch,ids,answers):
        ok=(a.lower() in s.lower()) if a else ("немає відповіді" in s.lower())
        cite=(str(g+1) in re.findall(r"\[(\d+)\]",s) and g in r) if a else None
        rows.append(dict(question=q,gold=a,answer=s,correct=ok,citation=cite,retrieved=r,hit=g in r if a else None))
df=pd.DataFrame(rows); display(df); print("Accuracy (автоматична перевірка ключових слів):",df.correct.mean(),"Citation:",df.citation.dropna().mean())
print("Час:",time.perf_counter()-start)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

,question,gold,answer,correct,citation,retrieved,hit
0,Як підготувати SQL-запит?,PDO::prepare,PDO. Підготовлене SQL-запит створює PDO::prepa...,True,False,"[0, 1, 3]",True
1,Як почати транзакцію?,PDO::beginTransaction,PDO::beginTransaction,True,False,"[1, 0, 6]",True
2,Яке кодування задавати в DSN?,utf8mb4,UTF-8 mb4,False,False,"[2, 1, 0]",True
3,Як хешувати пароль?,password_hash,[4],False,True,"[3, 2, 5]",True
4,Як екранувати HTML?,htmlspecialchars,Безпека. htmlspecialchars екранує спеціальні с...,True,False,"[4, 10, 5]",True
5,Як створити випадковий CSRF-токен?,random_bytes,random_bytes,True,False,"[5, 4, 2]",True
6,Як почати сесію?,session_start,[7],False,True,"[6, 0, 1]",True
7,Як оновити ID сесії після входу?,session_regenerate_id,session_regenerate_id(),True,False,"[7, 8, 6]",True
8,Де зберігаються змінні сесії у коді?,$_SESSION,$_SESSION,True,False,"[8, 7, 6]",True
9,Яка функція надсилає заголовок?,header,HTTP. header,True,False,"[9, 4, 0]",True


Accuracy (автоматична перевірка ключових слів): 0.76 Citation: 0.1
Час: 487.28382088499995


In [5]:
fail=df[(~df.correct)|(df.citation==False)]
display(fail)
print("Кількість відмов:",len(fail))
for _,r in fail.iterrows():
    print(r.question, "Промах ретриву: збільшити k або змінити пошук" if r.hit is False else "Перевірити генерацію, цитату та еталон вручну")

,question,gold,answer,correct,citation,retrieved,hit
0,Як підготувати SQL-запит?,PDO::prepare,PDO. Підготовлене SQL-запит створює PDO::prepa...,True,False,"[0, 1, 3]",True
1,Як почати транзакцію?,PDO::beginTransaction,PDO::beginTransaction,True,False,"[1, 0, 6]",True
2,Яке кодування задавати в DSN?,utf8mb4,UTF-8 mb4,False,False,"[2, 1, 0]",True
3,Як хешувати пароль?,password_hash,[4],False,True,"[3, 2, 5]",True
4,Як екранувати HTML?,htmlspecialchars,Безпека. htmlspecialchars екранує спеціальні с...,True,False,"[4, 10, 5]",True
5,Як створити випадковий CSRF-токен?,random_bytes,random_bytes,True,False,"[5, 4, 2]",True
6,Як почати сесію?,session_start,[7],False,True,"[6, 0, 1]",True
7,Як оновити ID сесії після входу?,session_regenerate_id,session_regenerate_id(),True,False,"[7, 8, 6]",True
8,Де зберігаються змінні сесії у коді?,$_SESSION,$_SESSION,True,False,"[8, 7, 6]",True
9,Яка функція надсилає заголовок?,header,HTTP. header,True,False,"[9, 4, 0]",True


Кількість відмов: 21
Як підготувати SQL-запит? Перевірити генерацію, цитату та еталон вручну
Як почати транзакцію? Перевірити генерацію, цитату та еталон вручну
Яке кодування задавати в DSN? Перевірити генерацію, цитату та еталон вручну
Як хешувати пароль? Перевірити генерацію, цитату та еталон вручну
Як екранувати HTML? Перевірити генерацію, цитату та еталон вручну
Як створити випадковий CSRF-токен? Перевірити генерацію, цитату та еталон вручну
Як почати сесію? Перевірити генерацію, цитату та еталон вручну
Як оновити ID сесії після входу? Перевірити генерацію, цитату та еталон вручну
Де зберігаються змінні сесії у коді? Перевірити генерацію, цитату та еталон вручну
Яка функція надсилає заголовок? Перевірити генерацію, цитату та еталон вручну
Де отримати поля POST-форми? Перевірити генерацію, цитату та еталон вручну
Де отримати параметри URL? Перевірити генерацію, цитату та еталон вручну
Який модифікатор приховує властивість від інших класів? Перевірити генерацію, цитату та еталон вруч

In [6]:
import gradio as gr
def ask(q):
    vec=emb.encode(q,normalize_embeddings=True).astype("float32"); ids=rank(q,vec,True)[:3]
    return generate([prompts(q,ids)])[0]+"\n\nДжерела:\n"+"\n".join(f"[{i+1}] {chunks[i]}" for i in ids)
gr.Interface(fn=ask,inputs="text",outputs="text",title="PHP RAG").launch(share=True,prevent_thread_lock=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://481ccf8b9dd854a54f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Звіт: власна RAG-система PHP
Авторський навчальний корпус: короткі нотатки PHP, CC0 (заявлена ліцензія), 20 фрагментів, 25 запитань (20 з відповіддю, 5 без). RAG дозволяє оновлювати знання без перенавчання і показувати джерела. MiniLM, нормалізовані вектори, FAISS, BM25 та RRF; фрагменти сформовані вручну за змістом, без перекриття.

| Метод | Hit@1 | Hit@5 | MRR |
|---|---:|---:|---:|
| Векторний | 0,90 | 0,95 | 0,932143 |
| Гібридний | 0,95 | 1,00 | 0,966667 |

Генератор Qwen2.5-1.5B-Instruct, greedy, 64 нові токени. Автоматична точність за ключовими словами 76%, цитування 10% серед питань з відповіддю. Час 487,28 с. Gradio запущено. Гібридний пошук покращив ретрив, але генератор не завжди виконує вимогу цитування. Автоматична перевірка ключових слів не є повною семантичною оцінкою.

Обмеження: фактичний запуск використовував Transformers напряму, а не OpenAI-сумісний API. Додатковий API-експеримент потребує доступного сервера та повторного оцінювання; наявні метрики не можна приписувати API.